In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mph
import mphsweepkit as msk
from mphsweepkit.plot_fields import PlotField

In [3]:
# Start the COMSOL client
client = mph.start()

In [4]:
# Load the model
model = client.load('core_cross_section_solved.mph')

Load an already solved CascadedSweepModel

In [5]:
csm = msk.CascadedSweepModel(model, 'Study on Cross-Sections', show_param_names=True)
# TODO: load_data_output_from_dir

Initialized CascadedSweepModel
Study name: Study on Cross-Sections
Sweep Structure:
    - Geometry Sweep (BatchSweep) -> 'hor_slit', 'vert_slit', 'w', 'l_r', 'a_e'
      - Material Sweep (MaterialSweep) -> 'matsw.comp1.core'
        - Excitation Sweep (Parametric) -> 'b_mean'
          - Frequency Sweep (Frequency)
--------------------------------
Data updated from MPh-model.
Input data shape: (156, 8)
Reset output data to shape of the input data: (156, 0)
Combined shape: (156, 8)


In [6]:
# 1) 
csm.print_available_selections()
print("\n")

# 2)
# the selection name must be chosen as defined in the COMSOL model.
selection_name = "Core"
selection_type = "dom"
selection_dataset_name = "Solution Core"

# 3)
selection_domain_tag = csm._get_selection_tag(selection_name, selection_type="dom")

print(f"Selected domain tag: {selection_domain_tag}")
print("\n")

# 4)
# Create a dataset from the selection
csm.create_dataset_selection(selection_name, selection_type, selection_dataset_name)
# remove a dataset:
# csm.node_datasets.children()[-1].remove()

Available selections:
geom1_csel1_pnt Core
geom1_csel1_bnd Core
geom1_csel1_dom Core
geom1_csel2_pnt Air
geom1_csel2_bnd Air
geom1_csel2_dom Air
geom1_csel4_pnt Vertical Slit
geom1_csel4_bnd Vertical Slit
geom1_csel4_dom Vertical Slit
geom1_csel5_pnt Horizontal Slit
geom1_csel5_bnd Horizontal Slit
geom1_csel5_dom Horizontal Slit
geom1_csel3_pnt Region
geom1_csel3_bnd Region
geom1_csel3_dom Region


Selected domain tag: geom1_csel1_dom


Creating dataset 'Solution Core' from selection 'Core' of type 'dom'.
Available datasets: ['Study on Cross-Sections//Solution 1', 'Cascaded Sweep Solution', 'Solution Core']


In [23]:
# 5)
# TODO: modify parameter picking
no_geometries = 3

for i in range(no_geometries):
    export_node = csm.export_dataset_with_expressions(
    dataset_name=selection_dataset_name,
    expressions=["mf.normB", "mf.normE"],
    descriptions=["Magnetic Flux Density", "Electric Field"],
    looplevel=[[5], [1, 2], [i+1]],
    looplevelinput=['all', 'all', 'manual']
    )

    print(f"Loop level: {export_node.property('looplevel')}")

Dataset to be exported:             'Solution Core'
Overwrite existing export node:     'Expressions on Solution Core'
List of expressions:                ['mf.normB', 'mf.normE']
Loop level: [array([5.]) array([1., 2.]) array([1.])]
Dataset to be exported:             'Solution Core'
Overwrite existing export node:     'Expressions on Solution Core'
List of expressions:                ['mf.normB', 'mf.normE']
Loop level: [array([5.]) array([1., 2.]) array([2.])]
Dataset to be exported:             'Solution Core'
Overwrite existing export node:     'Expressions on Solution Core'
List of expressions:                ['mf.normB', 'mf.normE']
Loop level: [array([5.]) array([1., 2.]) array([3.])]


Release the Client

In [24]:
client.remove(model)
client.clear()
client.disconnect()